In [ ]:
# ============================================================
# INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q \
    kagglehub \
    python-docx \
    nltk \
    sentence-transformers \
    faiss-cpu \
    groq \
    pandas

In [ ]:
# ============================================================
# DOWNLOAD KAGGLE DATASET
# ============================================================

import kagglehub

dataset_path = kagglehub.dataset_download(
    "valentynbovchaliuk/underdefense-iso27001-templates-policies"
)

print("Dataset downloaded to:")
print(dataset_path)

In [ ]:
# ============================================================
# EXPLORE DATASET
# ============================================================

import os

for root, dirs, files in os.walk(dataset_path):

    for file in files:
        print(os.path.join(root, file))

/root/.cache/kagglehub/datasets/valentynbovchaliuk/underdefense-iso27001-templates-policies/versions/1/UnderDefense MAXI - Asset management policy.docx
/root/.cache/kagglehub/datasets/valentynbovchaliuk/underdefense-iso27001-templates-policies/versions/1/UnderDefense MAXI - Change management policy.docx
/root/.cache/kagglehub/datasets/valentynbovchaliuk/underdefense-iso27001-templates-policies/versions/1/UnderDefense MAXI - Vulnerability management policy.docx
/root/.cache/kagglehub/datasets/valentynbovchaliuk/underdefense-iso27001-templates-policies/versions/1/UnderDefense MAXI - Risk management policy.docx
/root/.cache/kagglehub/datasets/valentynbovchaliuk/underdefense-iso27001-templates-policies/versions/1/UnderDefense MAXI - Data retention and destruction policy.docx
/root/.cache/kagglehub/datasets/valentynbovchaliuk/underdefense-iso27001-templates-policies/versions/1/UnderDefense MAXI - Password management policy.docx
/root/.cache/kagglehub/datasets/valentynbovchaliuk/underdefense

In [ ]:
# ============================================================
# READ DOCX DOCUMENTS
# ============================================================

from docx import Document
import os

documents = []

for root, dirs, files in os.walk(dataset_path):

    for file in files:

        if file.lower().endswith(".docx"):

            file_path = os.path.join(root, file)

            doc = Document(file_path)

            text = "\n".join(
                paragraph.text
                for paragraph in doc.paragraphs
                if paragraph.text.strip()
            )

            if text.strip():

                documents.append({
                    "filename": file,
                    "text": text
                })

print("Documents loaded:", len(documents))

In [ ]:
# ============================================================
# VIEW FIRST DOCUMENT
# ============================================================

print("FILE:")
print(documents[0]["filename"])

print("\nDOCUMENT TEXT:")
print(documents[0]["text"][:5000])

In [ ]:
# ============================================================
# TEXT CLEANING
# ============================================================

import re

def clean_text(text):

    # Replace multiple spaces with one space
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing spaces
    text = text.strip()

    return text


for document in documents:

    document["clean_text"] = clean_text(
        document["text"]
    )

print(documents[0]["clean_text"][:2000])

In [ ]:
# ============================================================
# TOKENIZATION DEMO
# ============================================================

import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.tokenize import word_tokenize

sample_text = documents[0]["clean_text"][:500]

tokens = word_tokenize(sample_text)

print("ORIGINAL TEXT:")
print(sample_text)

print("\nTOKENS:")
print(tokens)

print("\nNUMBER OF TOKENS:")
print(len(tokens))

In [ ]:
# ============================================================
# TEXT CHUNKING
# ============================================================

def create_chunks(text, chunk_size=800, overlap=150):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(
            words[start:end]
        )

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
# ============================================================
# CREATE DOCUMENT CHUNKS
# ============================================================

chunks = []

for document in documents:

    document_chunks = create_chunks(
        document["clean_text"],
        chunk_size=800,
        overlap=150
    )

    for chunk_number, chunk in enumerate(document_chunks):

        chunks.append({
            "chunk_id": len(chunks),
            "filename": document["filename"],
            "chunk_number": chunk_number,
            "text": chunk
        })

print("Total chunks:", len(chunks))

In [ ]:
# ============================================================
# CREATE DOCUMENT CHUNKS
# ============================================================

chunks = []

for document in documents:

    document_chunks = create_chunks(
        document["clean_text"],
        chunk_size=800,
        overlap=150
    )

    for chunk_number, chunk in enumerate(document_chunks):

        chunks.append({
            "chunk_id": len(chunks),
            "filename": document["filename"],
            "chunk_number": chunk_number,
            "text": chunk
        })

print("Total chunks:", len(chunks))

In [ ]:
# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

In [ ]:
# ============================================================
# GENERATE EMBEDDINGS
# ============================================================

chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

embeddings = embeddings.astype("float32")

print("Embedding shape:", embeddings.shape)

In [ ]:
# ============================================================
# VIEW EMBEDDING VECTOR
# ============================================================

print("CHUNK:")
print(chunks[0]["text"][:500])

print("\nVECTOR:")
print(embeddings[0])

print("\nVECTOR DIMENSION:")
print(len(embeddings[0]))

In [ ]:
# ============================================================
# CREATE FAISS VECTOR DATABASE
# ============================================================

import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS VECTOR DATABASE")
print("=" * 60)

print("Vector dimension :", dimension)
print("Vectors stored   :", index.ntotal)

In [ ]:
# ============================================================
# RETRIEVAL FUNCTION
# ============================================================

def retrieve_documents(query, top_k=5):

    # Convert query into embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS
    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for distance, idx in zip(
        distances[0],
        indices[0]
    ):

        results.append({
            "chunk_id": int(idx),
            "distance": float(distance),
            "filename": chunks[idx]["filename"],
            "text": chunks[idx]["text"]
        })

    return results

In [ ]:
# ============================================================
# TEST RETRIEVAL
# ============================================================

query = "What is the purpose of the information security policy?"

results = retrieve_documents(
    query,
    top_k=5
)

print("=" * 70)
print("USER QUESTION")
print("=" * 70)

print(query)

print("\n" + "=" * 70)
print("RETRIEVED CHUNKS")
print("=" * 70)

for rank, result in enumerate(results, 1):

    print(f"\nRESULT {rank}")
    print("-" * 70)

    print("Chunk ID :", result["chunk_id"])
    print("Distance:", round(result["distance"], 4))
    print("File    :", result["filename"])

    print("\nText:")
    print(result["text"][:800])

In [ ]:
# ============================================================
#  CONNECT GROQ LLM
# ============================================================

from groq import Groq

# Add your Groq API key here
GROQ_API_KEY = "your_api_key"

client = Groq(api_key=GROQ_API_KEY)

MODEL = "llama-3.3-70b-versatile"

print("Groq client initialized!")
print("Model:", MODEL)

In [ ]:
print("\n--- Verifying Groq API key and permissions ---")

try:
    # Attempt to list models to verify API key and permissions
    models = client.models.list()
    print("API Key is valid. Accessible models:")
    for model in models.data:
        print(f"- {model.id}")
    if not models.data:
        print("No models are currently accessible with this API key. Please check your Groq account for available models and permissions.")
except Exception as e:
    print(f"Error verifying API key or permissions: {e}")
    print("Please ensure your GROQ_API_KEY is correct and has the necessary permissions.")

In [ ]:
# ============================================================
#  BUILD RAG CONTEXT
# ============================================================

def build_context(results):

    context = ""

    for i, result in enumerate(results, 1):

        context += f"""
--- DOCUMENT {i} ---
File: {result['filename']}

{result['text']}
"""

    return context


# Test
context = build_context(results)

print(context[:5000])

In [ ]:
# ============================================================
#  RAG QUESTION ANSWERING
# ============================================================

def ask_rag(question, top_k=5):

    # --------------------------------------------------------
    # 1. Retrieve relevant documents from FAISS
    # --------------------------------------------------------

    results = retrieve_documents(
        question,
        top_k=top_k
    )

    # --------------------------------------------------------
    # 2. Build context from retrieved chunks
    # --------------------------------------------------------

    context = build_context(results)

    # --------------------------------------------------------
    # 3. Create prompt for the LLM
    # --------------------------------------------------------

    prompt = f"""
You are an information security policy assistant.

Answer the user's question using ONLY the information
provided in the context below.

If the answer cannot be found in the context,
say:

"I could not find this information in the provided documents."

Do not invent or assume information.

CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    # --------------------------------------------------------
    # 4. Send prompt to Groq
    # --------------------------------------------------------

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You answer questions using provided document context."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    # --------------------------------------------------------
    # 5. Extract answer
    # --------------------------------------------------------

    answer = response.choices[0].message.content

    return answer, results

In [ ]:
# ============================================================
# STEP 19: TEST THE COMPLETE RAG SYSTEM
# ============================================================

question = "What is the purpose of the information security policy?"

answer, retrieved_results = ask_rag(
    question,
    top_k=5
)

print("=" * 80)
print("QUESTION")
print("=" * 80)

print(question)

print("\n" + "=" * 80)
print("RAG ANSWER")
print("=" * 80)

print(answer)

In [ ]:
# ============================================================
# SHOW RETRIEVED SOURCES
# ============================================================

print("=" * 70)
print("RETRIEVED INFORMATION")
print("=" * 70)

for rank, result in enumerate(retrieved_results, 1):

    print(f"\nSOURCE {rank}")
    print("-" * 70)

    print("File:", result["filename"])
    print("Chunk ID:", result["chunk_id"])
    print("Distance:", round(result["distance"], 4))

    print("\nText:")
    print(result["text"][:1000])

In [ ]:
questions = [
    "What is the purpose of information security?",
    "What are the responsibilities of employees regarding information security?",
    "What should employees do when they identify a security incident?",
    "What are the requirements for access control?",
    "How should sensitive information be protected?"
]

for question in questions:

    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("=" * 80)

    answer, _ = ask_rag(question)

    print(answer)